# 01 — Simulation Demo

This notebook demonstrates the core atmospheric simulation and SH-WFS
sensor pipeline:

1. Load configuration
2. Generate Kolmogorov and von Karman phase screens and plot them
3. Animate frozen-flow evolution of a multi-layer atmosphere (5 frames)
4. Simulate the SH-WFS: plot the slope field and spot pattern
5. Show the effect of noise on centroids (clean vs noisy)
6. Generate a mini dataset (100 frames) and show basic statistics
   (r0 estimation, structure function)

All cells run end-to-end without errors.


In [1]:
import sys
sys.path.insert(0, "..")

import numpy as np
import matplotlib.pyplot as plt
import yaml

from sim.turbulence import (
    kolmogorov_psd, von_karman_psd, generate_phase_screen,
    MultiLayerAtmosphere, build_atmosphere_from_config,
)
from sim.phase_screen import apply_aperture_mask, get_aperture_mask, compute_zernike_coefficients
from sim.shwfs import SHWFSSensor
from sim.noise import add_photon_noise, add_readout_noise, centroid_cog
from sim.dataset_gen import generate_dataset, load_dataset
from viz.plot_utils import plot_phase_map, plot_slope_field, plot_spot_pattern

with open("../config.yaml") as f:
    config = yaml.safe_load(f)

sim_cfg = config["sim"]
turb_cfg = config["turbulence"]
N = sim_cfg["grid_size"]
pixel_scale = sim_cfg["aperture_diameter_m"] / N
print(f"Grid size N={N}, pixel_scale={pixel_scale:.4f} m, r0={turb_cfg['r0_m']} m, L0={turb_cfg['L0_m']} m")


Grid size N=128, pixel_scale=0.0039 m, r0=0.15 m, L0=25.0 m


## 1. Phase screen generation

Generate single phase screens using both the Kolmogorov and von Karman
power spectral densities, and plot them side by side.


In [2]:
r0 = turb_cfg["r0_m"]
L0 = turb_cfg["L0_m"]

screen_kolmogorov = generate_phase_screen(N, pixel_scale, r0, L0, model="kolmogorov", seed=1)
screen_vonkarman = generate_phase_screen(N, pixel_scale, r0, L0, model="von_karman", seed=1)

mask = get_aperture_mask(N, "circular")

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
plot_phase_map(apply_aperture_mask(screen_kolmogorov, "circular"), "Kolmogorov phase screen", mask=mask, ax=axes[0])
plot_phase_map(apply_aperture_mask(screen_vonkarman, "circular"), "Von Karman phase screen", mask=mask, ax=axes[1])
plt.tight_layout()
plt.show()

print(f"Kolmogorov screen RMS (rad): {np.std(screen_kolmogorov[mask]):.3f}")
print(f"Von Karman screen RMS (rad): {np.std(screen_vonkarman[mask]):.3f}")


Kolmogorov screen RMS (rad): 1.054
Von Karman screen RMS (rad): 1.022


## 2. Frozen-flow evolution of a multi-layer atmosphere

Build a `MultiLayerAtmosphere` from the config and evolve it for 5
frames, plotting the integrated phase at each step.


In [3]:
atmosphere = build_atmosphere_from_config(config, N=N, pixel_scale=pixel_scale, seed=42)

n_evolution_frames = 5
fig, axes = plt.subplots(1, n_evolution_frames, figsize=(4 * n_evolution_frames, 4))

for k in range(n_evolution_frames):
    phase = atmosphere.get_integrated_phase_radians()
    phase_masked = apply_aperture_mask(phase, "circular")
    plot_phase_map(phase_masked, f"t = {k} ms", mask=mask, colorbar=False, ax=axes[k])
    atmosphere.evolve(sim_cfg["dt_s"])

plt.tight_layout()
plt.show()


## 3. SH-WFS simulation: slope field and spot pattern

Propagate the current integrated phase through the SH-WFS sensor and
visualize the resulting slope field and a single subaperture's spot
pattern (clean, ideal centroid vs reference).


In [4]:
sensor = SHWFSSensor(
    n_subapertures=sim_cfg["n_subapertures"],
    pixels_per_subaperture=sim_cfg["detector_pixels_per_subaperture"],
    focal_length=sim_cfg["focal_length_m"],
    pitch=sim_cfg["mla_pitch_m"],
    wavelength=sim_cfg["wavelength_m"],
)

phase = atmosphere.get_integrated_phase_radians()
phase_masked = apply_aperture_mask(phase, "circular")

slopes_x, slopes_y = sensor.propagate(phase_masked)
valid_mask = sensor.get_valid_subaperture_mask()

fig, ax = plt.subplots(figsize=(6, 6))
plot_slope_field(slopes_x, slopes_y, valid_mask, ax=ax)
plt.show()

print(f"Number of valid subapertures: {valid_mask.sum()} / {valid_mask.size}")
print(f"Slope x range: [{slopes_x[valid_mask].min():.4f}, {slopes_x[valid_mask].max():.4f}] rad")


Number of valid subapertures: 80 / 100
Slope x range: [-0.3967, 0.1090] rad


In [5]:
# Spot pattern for a single subaperture with non-zero tilt
i0, j0 = np.argwhere(valid_mask)[len(np.argwhere(valid_mask)) // 2]
sx, sy = slopes_x[i0, j0], slopes_y[i0, j0]

spot_clean = sensor.simulate_spot_image(sx, sy)

ref_spots = sensor.generate_reference_spots()
reference_grid = ref_spots[i0:i0+1, j0:j0+1]

n = sensor.pix_per_sub
ref_cx, ref_cy = (n - 1) / 2.0, (n - 1) / 2.0
cx, cy = centroid_cog(spot_clean)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(spot_clean, origin="lower", cmap="inferno")
axes[0].scatter([ref_cx], [ref_cy], marker="x", c="white", label="reference")
axes[0].scatter([cx], [cy], marker=".", c="cyan", label="measured")
axes[0].legend()
axes[0].set_title(f"Subaperture ({i0},{j0}) spot image (clean)")

axes[1].bar(["slope_x", "slope_y"], [sx, sy])
axes[1].set_title("Local slope (rad)")
plt.tight_layout()
plt.show()

print(f"Reference centroid: ({ref_cx:.2f}, {ref_cy:.2f})")
print(f"Measured centroid:  ({cx:.2f}, {cy:.2f})")


Reference centroid: (3.50, 3.50)
Measured centroid:  (3.49, 4.00)


## 4. Effect of noise on centroids

Compare clean vs noisy spot images (photon + readout noise) and their
centroids.


In [6]:
noise_cfg = config["noise"]
flux = noise_cfg["flux_photons_per_frame"]
readout = noise_cfg["readout_noise_e"]

np.random.seed(0)
spot_noisy = add_photon_noise(spot_clean, flux)
spot_noisy = add_readout_noise(spot_noisy, readout)
spot_noisy = np.clip(spot_noisy, 0, None)

cx_noisy, cy_noisy = centroid_cog(spot_noisy)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(spot_clean, origin="lower", cmap="inferno")
axes[0].scatter([cx], [cy], marker="x", c="cyan")
axes[0].set_title(f"Clean spot, centroid=({cx:.2f},{cy:.2f})")

axes[1].imshow(spot_noisy, origin="lower", cmap="inferno")
axes[1].scatter([cx_noisy], [cy_noisy], marker="x", c="cyan")
axes[1].set_title(f"Noisy spot, centroid=({cx_noisy:.2f},{cy_noisy:.2f})")

plt.tight_layout()
plt.show()

print(f"Centroid shift due to noise: ({cx_noisy - cx:.3f}, {cy_noisy - cy:.3f}) pixels")


Centroid shift due to noise: (0.003, 0.042) pixels


## 5. Mini dataset generation and statistics

Generate a small 100-frame dataset and check basic statistics: r0
estimation from the Zernike variance spectrum, and the wavefront
structure function.


In [7]:
import os
os.makedirs("../data", exist_ok=True)

mini_dataset_path = "../data/mini_dataset.h5"
generate_dataset(config, n_frames=100, output_path=mini_dataset_path, seed=123)

data = load_dataset(mini_dataset_path)
print("Dataset shapes:")
print(f"  slopes:         {data['slopes'].shape}")
print(f"  zernike_coeffs: {data['zernike_coeffs'].shape}")
print(f"  phase_maps:     {data['phase_maps'].shape}")


Generating dataset:   0%|          | 0/100 [00:00<?, ?it/s]

Generating dataset:   1%|          | 1/100 [00:00<00:10,  9.35it/s]

Generating dataset:   4%|▍         | 4/100 [00:00<00:05, 16.95it/s]

Generating dataset:   7%|▋         | 7/100 [00:00<00:04, 18.77it/s]

Generating dataset:  10%|█         | 10/100 [00:00<00:04, 19.50it/s]

...

Generating dataset:  94%|█████████▍| 94/100 [00:04<00:00, 20.61it/s]

Generating dataset:  97%|█████████▋| 97/100 [00:04<00:00, 20.66it/s]

Generating dataset: 100%|██████████| 100/100 [00:04<00:00, 20.64it/s]

Generating dataset: 100%|██████████| 100/100 [00:04<00:00, 20.43it/s]

Dataset shapes:
  slopes:         (100, 2, 10, 10)
  zernike_coeffs: (100, 36)
  phase_maps:     (100, 128, 128)


In [8]:
from temporal.turbulence_param import estimate_r0_from_zernike

D = sim_cfg["aperture_diameter_m"]
r0_est = estimate_r0_from_zernike(data["zernike_coeffs"], wavelength=sim_cfg["wavelength_m"], D=D)
print(f"True r0:      {turb_cfg['r0_m']:.4f} m")
print(f"Estimated r0: {r0_est:.4f} m")

# Structure function D(r) for small separations
phase_sample = data["phase_maps"][0]
for r_pix in [1, 2, 4]:
    r_m = r_pix * pixel_scale
    diffs = phase_sample[:, r_pix:] - phase_sample[:, :-r_pix]
    D_measured = np.mean(diffs ** 2)
    D_theory = 6.88 * (r_m / turb_cfg["r0_m"]) ** (5.0 / 3.0)
    print(f"r={r_m*1000:.1f} mm: D_measured={D_measured:.3f}, D_theory={D_theory:.3f}")


True r0:      0.1500 m
Estimated r0: 0.2191 m
r=3.9 mm: D_measured=0.807, D_theory=0.016
r=7.8 mm: D_measured=1.601, D_theory=0.050
r=15.6 mm: D_measured=2.878, D_theory=0.159


In [9]:
# Zernike variance spectrum across the mini dataset
variances = np.var(data["zernike_coeffs"], axis=0)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(np.arange(1, len(variances) + 1), variances)
ax.set_xlabel("Noll index j")
ax.set_ylabel("Variance (rad^2)")
ax.set_title("Zernike coefficient variance spectrum (mini dataset)")
ax.set_yscale("log")
plt.show()
